# 04 — Model 1: Linear Baseline (raw logistic regression) on Sleep-EDF

**Bottom rung of the complexity ladder** — multinomial logistic regression on the raw, flattened
30-second epoch (3000 samples → 5 sleep stages), no feature engineering, no class weighting, no
rebalancing. It doubles as a directly-inspectable interpretability anchor for validating the XAI
machinery later.

**What's new in this notebook: the reusable training driver.** Every model on the ladder (this linear
baseline, the three CNNs, the transformer) is trained through **one shared helper**,
`sleep_edf.training.run_all_seeds`, so the timing / progress / incremental-save behaviour is identical
everywhere. We build it here on the *fast* linear baseline so it's tested before it matters for the slow
neural models. The driver:

1. **Times the first seed** and prints a total-time estimate — then **continues automatically** (it is
   *not* a prompt; you launch once and walk away).
2. Trains **all remaining seeds to completion**, unattended.
3. Shows a **tqdm progress bar** — an outer bar over seeds, and (for the neural models later) an inner
   bar over epochs via a `tick` callback. Logistic regression has no epochs, so only the seed bar shows.
4. **Saves each seed's results to disk the moment it finishes** — an interruption never loses completed
   seeds, and a re-run can `resume`.
5. Aggregates, prints a per-seed + mean±std summary, and saves the aggregate.

> **How to run this notebook:** run the setup cells top-to-bottom, then run the single **LAUNCH TRAINING**
> cell (clearly marked below) — that one cell trains all 5 seeds unattended. The reporting cells after it
> read the saved results.

In [1]:
import sys
from pathlib import Path
# Locate repo root robustly: walk up to the dir containing sleep_edf/ (depth-independent).
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "sleep_edf").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, precision_recall_fscore_support)

from sleep_edf.loader import load_sleep_edf              # 20K subsample by default (train)
import sleep_edf.config as cfg                           # INPUT_LENGTH=3000, IN_CHANNELS=1, N_CLASSES=5
from sleep_edf.training import run_all_seeds             # the shared timed/progress/save driver

CLASS_NAMES = ["W", "N1", "N2", "N3", "REM"]
SEEDS       = [0, 1, 2, 3, 4]
C_L2        = 1.0                 # inverse L2 strength (sklearn default)
MODEL_NAME  = "model1_linear"

OUT_DIR = PROJECT_ROOT / "sleep_edf" / "results" / "metrics"   # per-seed + aggregate results
FIG_DIR = PROJECT_ROOT / "sleep_edf" / "results" / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)
np.set_printoptions(precision=3, suppress=True)
print("config:", f"INPUT_LENGTH={cfg.INPUT_LENGTH}", f"IN_CHANNELS={cfg.IN_CHANNELS}",
      f"N_CLASSES={cfg.N_CLASSES}")
print("results ->", OUT_DIR)

config: INPUT_LENGTH=3000 IN_CHANNELS=1 N_CLASSES=5
results -> /Users/luciepasquier/Desktop/IMPERIAL/SUMMER THESIS/Thesis-Repo/sleep_edf/results/metrics


## 1. Load the data — the 20K training subsample, full test set

`load_sleep_edf("train")` returns the **20,000-epoch stratified subsample by default** (seed 42) — every
model on the ladder trains on this identical set. It prints a one-line record of that. `load_sleep_edf("test")`
returns the **full** test set (40,145 epochs), never subsampled. We also print the class balance and the
**majority-class floor** (accuracy you get for free by always predicting the largest class) as reference
lines for later.

In [2]:
X_train, y_train = load_sleep_edf("train")    # -> logs "[sleep_edf] training load: 20,000-epoch ..."
X_test,  y_test  = load_sleep_edf("test")     # full 40,145 (never subsampled)

assert X_train.shape == (20000, cfg.INPUT_LENGTH), X_train.shape
assert X_test.shape[1] == cfg.INPUT_LENGTH
print("X_train:", X_train.shape, X_train.dtype, "| X_test:", X_test.shape)

print(f"\n{'stage':<6}{'train n':>9}{'train %':>9}{'test n':>9}{'test %':>9}")
print("-" * 42)
for c in range(cfg.N_CLASSES):
    ntr, nte = int((y_train == c).sum()), int((y_test == c).sum())
    print(f"{CLASS_NAMES[c]:<6}{ntr:>9}{100*ntr/len(y_train):>8.1f}%{nte:>9}{100*nte/len(y_test):>8.1f}%")

test_counts = np.bincount(y_test, minlength=cfg.N_CLASSES)
floor_cls = int(test_counts.argmax()); floor = test_counts[floor_cls] / len(y_test)
chance_balanced = 1.0 / cfg.N_CLASSES
print("-" * 42)
print(f"majority-class floor (always predict {CLASS_NAMES[floor_cls]}): {floor:.4f} "
      f"({100*floor:.1f}% of test)   |   balanced-acc chance = {chance_balanced:.3f}")

[sleep_edf] training load: 20,000-epoch stratified subsample (seed 42)
X_train: (20000, 3000) float32 | X_test: (40145, 3000)

stage   train n  train %   test n   test %
------------------------------------------
W          6822    34.1%    12967    32.3%
N1         2185    10.9%     4551    11.3%
N2         7011    35.1%    14679    36.6%
N3         1329     6.6%     2716     6.8%
REM        2653    13.3%     5232    13.0%
------------------------------------------
majority-class floor (always predict N2): 0.3656 (36.6% of test)   |   balanced-acc chance = 0.200


## 2. The shared training driver, and Model 1's `train_one_seed`

The driver `run_all_seeds(train_one_seed, seeds, out_dir, model_name, ...)` is model-agnostic — it only
needs a function that trains **one** seed and returns a results dict:

```
train_one_seed(seed, tick=None) -> dict
```

`tick(done, total)` is the inner-epoch progress hook the neural models will call once per epoch; logistic
regression has no epochs, so Model 1 just ignores `tick` (only the outer seed bar shows). This keeps the
**interface constant across the whole ladder**.

The next cell **standardises the features once** (fit on train only — no leakage; seed-independent, so
doing it once is exact) and defines Model 1's `train_one_seed`. Running this cell fits a scaler and defines
a function — **it does not train the model** (that happens in the LAUNCH cell).

In [ ]:
# Standardise once: fit on TRAIN only (no leakage), transform both. Seed-independent.
scaler = StandardScaler().fit(X_train)
Xtr = scaler.transform(X_train).astype(np.float32)
Xte = scaler.transform(X_test).astype(np.float32)

def train_one_seed(seed, tick=None):
    # Logistic regression is convex + lbfgs is deterministic, so there is no epoch loop and `tick`
    # is unused here (only the outer seed bar shows). The same signature carries to the CNNs/transformer,
    # which WILL call tick(epoch+1, n_epochs) each epoch to drive an inner bar.
    clf = LogisticRegression(C=C_L2, penalty="l2", solver="lbfgs",
                             max_iter=1000, random_state=seed, n_jobs=-1)
    clf.fit(Xtr, y_train)
    yp = clf.predict(Xte)
    pr, rc, f1c, sup = precision_recall_fscore_support(
        y_test, yp, labels=list(range(cfg.N_CLASSES)), zero_division=0)
    return {
        "accuracy":          float(accuracy_score(y_test, yp)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, yp)),
        "macro_f1":          float(f1_score(y_test, yp, average="macro")),
        "n_params":          int(clf.coef_.size + clf.intercept_.size),
        "converged":         bool(int(clf.n_iter_[0]) < 1000),
        "confusion":         confusion_matrix(y_test, yp, labels=list(range(cfg.N_CLASSES))).tolist(),
        "per_class":         {CLASS_NAMES[c]: {"precision": float(pr[c]), "recall": float(rc[c]),
                                               "f1": float(f1c[c]), "support": int(sup[c])}
                              for c in range(cfg.N_CLASSES)},
    }

print("train_one_seed defined; features standardised", Xtr.shape, "-> ready to launch (next cell).")

## 3. ▶ LAUNCH TRAINING — run this one cell (unattended)

**This is the cell you run to train all 5 seeds.** It times seed 0, prints an estimate, then trains the
rest to completion automatically — no further input needed. Each seed is saved to
`sleep_edf/results/metrics/model1_linear_seed{seed}.json` as it finishes, and the aggregate to
`model1_linear_aggregate.json`.

`resume=True` means re-running skips seeds already on disk (so an interrupted run continues). To force a
clean re-train, delete `model1_linear_seed*.json` first (or pass `resume=False`).

In [ ]:
# ▶▶▶ LAUNCH: trains all seeds unattended (times seed 0, estimates, then continues automatically) ◀◀◀
aggregate = run_all_seeds(
    train_one_seed, SEEDS, OUT_DIR, MODEL_NAME,
    summary_keys=["accuracy", "balanced_accuracy", "macro_f1"],
    resume=True,          # skip seeds already saved to disk; delete the json files to re-train fresh
)

## 4. Did it learn? — floor, seed variance, and the minority classes

The cells below read the **saved** results (so they work after the launch cell, and after a kernel
restart). We check overall + balanced accuracy against the majority floor and chance, seed-to-seed
variance, and — the point of this baseline — **per-class** performance, especially the minority classes
**N1** and **N3** (expected weak; we want that visible, not hidden behind an okay-looking overall number).

In [ ]:
import json
agg = json.load(open(OUT_DIR / f"{MODEL_NAME}_aggregate.json"))
per_seed = agg["per_seed"]; s = agg["summary"]

acc_m, acc_sd = s["accuracy"]["mean"], s["accuracy"]["std"]
bacc_m, bacc_sd = s["balanced_accuracy"]["mean"], s["balanced_accuracy"]["std"]
n_params = per_seed[str(SEEDS[0])]["n_params"]
print(f"parameter count (ladder low point): {n_params:,}")
print(f"accuracy      : {acc_m:.4f} ± {acc_sd:.4f}   (majority floor {floor:.4f};  margin {acc_m-floor:+.4f})")
print(f"balanced acc  : {bacc_m:.4f} ± {bacc_sd:.4f}   (chance {chance_balanced:.3f};  margin {bacc_m-chance_balanced:+.4f})")
print(f"macro-F1      : {s['macro_f1']['mean']:.4f} ± {s['macro_f1']['std']:.4f}")
print(f"seed spread (accuracy): {s['accuracy']['values']}")

# Per-class precision/recall/F1, averaged across seeds (support is per-seed identical).
print(f"\n{'stage':<6}{'precision':>11}{'recall':>9}{'f1':>9}{'support':>10}")
print("-" * 45)
for cn in CLASS_NAMES:
    P = np.mean([per_seed[str(sd)]["per_class"][cn]["precision"] for sd in SEEDS])
    R = np.mean([per_seed[str(sd)]["per_class"][cn]["recall"]    for sd in SEEDS])
    F = np.mean([per_seed[str(sd)]["per_class"][cn]["f1"]        for sd in SEEDS])
    sup = per_seed[str(SEEDS[0])]["per_class"][cn]["support"]
    print(f"{cn:<6}{P:>11.3f}{R:>9.3f}{F:>9.3f}{sup:>10}")
print("\nWatch N1 and N3 (rarest): recall here shows how often the baseline actually finds them.")

In [ ]:
# Confusion matrix (summed across seeds) — counts and row-normalised (per-true-stage recall).
cm = np.sum([np.array(per_seed[str(sd)]["confusion"]) for sd in SEEDS], axis=0)
cmn = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, M, title, fmt in [(ax[0], cm, "Confusion (counts, summed over seeds)", "d"),
                         (ax[1], cmn, "Row-normalised (recall per true stage)", ".2f")]:
    im = a.imshow(M, cmap="Blues", vmin=0, vmax=(M.max() if fmt=="d" else 1))
    for i in range(5):
        for j in range(5):
            a.text(j, i, format(M[i, j], fmt), ha="center", va="center", fontsize=8,
                   color="white" if M[i, j] > (M.max()*0.5 if fmt=="d" else 0.5) else "black")
    a.set_xticks(range(5)); a.set_xticklabels(CLASS_NAMES)
    a.set_yticks(range(5)); a.set_yticklabels(CLASS_NAMES)
    a.set_xlabel("predicted"); a.set_ylabel("true"); a.set_title(title)
fig.tight_layout()
fig.savefig(FIG_DIR / "sleep_edf_04_model1_confusion.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved:", (FIG_DIR / "sleep_edf_04_model1_confusion.png").relative_to(PROJECT_ROOT))

## 5. Verdict

Fill in from the numbers above once trained. Expected shape (from the earlier full-set run): the linear
baseline sits only **just above the majority floor** on accuracy and **just above chance** on balanced
accuracy — it leans on the two big stages (W, N2) and is **largely blind to the minorities** (N1, N3 near
zero recall). That is the honest weak bottom rung: a real-but-poor signal to attribute later, with the
N3 scarcity caveat carried forward. Seed-to-seed variance is ~0 (logistic regression is convex, so the
5 seeds converge to essentially the same fit) — the informative seed-variance story starts with the CNNs.